# APN on PhysioNet 2012: reproducing irregular time-series forecasting in PyHealth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunlabuiuc/PyHealth/blob/master/examples/apn_physionet2012_forecasting.ipynb)

This notebook reproduces the PhysioNet 2012 result of

> Xvyuan Liu, Xiangfei Qiu, Xingjian Wu, Zhengyu Li, Chenjuan Guo, Jilin Hu, Bin Yang.
> **Rethinking Irregular Time Series Forecasting: A Simple yet Effective Baseline.** AAAI 2026.
> [arXiv:2505.11250](https://arxiv.org/abs/2505.11250) · [authors' code](https://github.com/decisionintelligence/APN)

with PyHealth's `APN` model, `PhysioNet2012Dataset` and `PhysioNet2012Forecasting` task.
PhysioNet 2012 is open access (about 20 MB), so the whole experiment runs without credentials.
A Colab CPU runtime is enough; a GPU makes it faster.

**Setup (as in the paper's benchmark).** From the first 36 hours of an ICU stay, forecast the values of
36 vital signs and labs observed at the next 3 observation times. Values are hourly means, z-scored per
variable, with values beyond 5 standard deviations removed; the metric is the MSE (and MAE) over the observed
targets of the test records, in z-score units. Records are split 81/9/10 by `RecordID` order, as in the benchmark code.

**Paper's numbers on PhysioNet 2012** (Table 2, mean over 5 runs): APN MSE **0.3093**, MAE **0.3650**.
For context, the best baselines there: Warpformer MSE 0.3056, GraFITi MAE 0.3637.

## 1. Install PyHealth

APN is on PyHealth's master branch and will be in the next PyPI release. Until then, install from GitHub.

In [ ]:
!pip install -q "git+https://github.com/sunlabuiuc/PyHealth"

## 2. Download PhysioNet 2012 and build the dataset

`download=True` fetches sets a, b and c and their outcomes from PhysioNet into `root`; the first run also
builds PyHealth's event tables (a few minutes).

In [ ]:
import numpy as np
import torch

from pyhealth.datasets import PhysioNet2012Dataset, get_dataloader
from pyhealth.models import APN
from pyhealth.tasks import (
    PhysioNet2012Forecasting,
    physionet2012_benchmark_split,
    physionet2012_statistics,
)

ROOT = "physionet2012"          # raw files and PyHealth's tables go here
CACHE = "physionet2012_cache"   # PyHealth's processed cache
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

dataset = PhysioNet2012Dataset(root=ROOT, download=True, cache_dir=CACHE, num_workers=2)
dataset.stats()

## 3. The forecasting task and the benchmark split

The benchmark standardizes with statistics over all records (including the test records). To use the
training records only, pass their IDs: `physionet2012_statistics(dataset, patient_ids=...)`.

In [ ]:
mean, std = physionet2012_statistics(dataset)
samples = dataset.set_task(PhysioNet2012Forecasting(mean=mean, std=std), num_workers=2)
train, val, test = physionet2012_benchmark_split(samples)
print(f"{len(samples)} records: {len(train)} train, {len(val)} validation, {len(test)} test")

sample = samples[0]
print({k: tuple(v.shape) for k, v in sample.items() if hasattr(v, "shape")})
print("observed input values:", int(torch.isfinite(sample["x"]).sum()),
      "| observed targets:", int(torch.isfinite(sample["y"]).sum()))

## 4. A sanity baseline: last observation carried forward

Each variable's last value in the first 36 hours forecasts all 3 steps (the mean, 0, if it was never observed).

In [ ]:
def score(predict, loader):
    """MSE and MAE over all observed targets, pooled as in the benchmark code."""
    se = ae = n = 0.0
    for batch in loader:
        y = batch["y"].to(DEVICE)
        pred = predict(batch)
        mask = torch.isfinite(y)
        d = pred[mask] - y[mask]
        se += float((d ** 2).sum()); ae += float(d.abs().sum()); n += d.numel()
    return se / n, ae / n

def locf(batch):
    x = batch["x"].to(DEVICE)
    last = torch.zeros(x.shape[0], x.shape[2], device=DEVICE)
    for t in range(x.shape[1]):
        last = torch.where(torch.isfinite(x[:, t]), x[:, t], last)
    return last.unsqueeze(1).expand(-1, batch["y"].shape[1], -1)

test_loader = get_dataloader(test, batch_size=256)
locf_mse, locf_mae = score(locf, test_loader)
print(f"LOCF  test MSE {locf_mse:.4f}  MAE {locf_mae:.4f}")

## 5. Train APN with the paper's protocol

The paper's PhysioNet 2012 setting: hidden size 24, 20 patches, time encoding of size 8, dropout 0.1,
Adam with learning rate 0.03 (constant for 2 epochs, then x0.8 per epoch), batch size 32, up to 200 epochs with
early stopping after 10 epochs without a better validation MSE, and the mean over 5 runs.
Set `SEEDS = [0, 1, 2, 3, 4]` for the paper's 5 runs (one run takes several minutes on a CPU).

PyHealth's `Trainer` also trains APN (`Trainer(model, metrics=["mse", "mae"]).train(...)`), with a constant
learning rate; the loop below adds the paper's schedule.

In [ ]:
import copy

SEEDS = [0]
MAX_EPOCHS = 200

def train_apn(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = APN(train, hidden_dim=24, num_patches=20, time_dim=8, dropout=0.1).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.03)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: 1.0 if e < 2 else 0.8 ** (e - 2))
    train_loader = get_dataloader(train, batch_size=32, shuffle=True)
    val_loader = get_dataloader(val, batch_size=256)
    best, best_state, wait = float("inf"), None, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for batch in train_loader:
            opt.zero_grad()
            model(**batch)["loss"].backward()
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            val_mse, _ = score(lambda b: model(**b)["forecast"], val_loader)
        if val_mse < best:
            best, best_state, wait = val_mse, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= 10:
                break
    model.load_state_dict(best_state)
    print(f"seed {seed}: stopped after epoch {epoch}, best validation MSE {best:.4f}")
    return model

results = []
for seed in SEEDS:
    model = train_apn(seed)
    model.eval()
    with torch.no_grad():
        results.append(score(lambda b: model(**b)["forecast"], test_loader))
    print(f"seed {seed}: test MSE {results[-1][0]:.4f}  MAE {results[-1][1]:.4f}")

## 6. Results next to the paper

In [ ]:
import pandas as pd

res = np.array(results)
rows = [
    ("APN, paper (5 runs)", 0.3093, 0.3650),
    ("Warpformer, paper (best MSE baseline)", 0.3056, None),
    ("GraFITi, paper (best MAE baseline)", None, 0.3637),
    (f"APN, PyHealth ({len(SEEDS)} run{'s' if len(SEEDS) > 1 else ''})", res[:, 0].mean(), res[:, 1].mean()),
    ("Last observation carried forward, PyHealth", locf_mse, locf_mae),
]
table = pd.DataFrame(rows, columns=["Model", "Test MSE", "Test MAE"])
if len(SEEDS) > 1:
    print(f"APN over {len(SEEDS)} runs: MSE {res[:, 0].mean():.4f} ± {res[:, 0].std():.4f}, "
          f"MAE {res[:, 1].mean():.4f} ± {res[:, 1].std():.4f}")
table.round(4)

### Our run (local CPU, 5 seeds, this notebook's protocol)

| Model | Test MSE | Test MAE |
|---|---|---|
| APN, paper (5 runs) | 0.3093 | 0.3650 |
| **APN, PyHealth (5 runs, seeds 0-4)** | **0.3047 ± 0.0006** | **0.3710 ± 0.0006** |
| Last observation carried forward | 0.4163 | 0.4115 |

Per seed (MSE / MAE): 0.3057 / 0.3717, 0.3042 / 0.3712, 0.3040 / 0.3700, 0.3044 / 0.3704, 0.3052 / 0.3715.
Each run stopped early, after 36 to 47 epochs. The MSE is slightly lower and the MAE slightly higher than the paper's;
the differences in the data pipeline listed below are a likely cause.

## 7. Notes for the authors: what to check

- **Model** (`pyhealth/models/apn.py`): a reimplementation from the paper. Time-Aware Patch Aggregation with
  learnable left offsets and log-widths per variable and patch, a learned per-variable temperature, soft
  membership `sigmoid((right - t)/tau) * sigmoid((t - left)/tau)` times the observation mask, a weight-normalized
  average of [value, time encoding] per patch, a linear projection with a residual feed-forward layer and
  LayerNorm, sinusoidal patch positions, a learned per-variable query attending over the patches, and a two-layer
  MLP decoder on [summary, encoding of the query time].
- **Data** (`PhysioNet2012Forecasting`): hourly bins, the 36 variables of the benchmark (Weight excluded), z-scores
  over all records, values at or beyond 5 SD removed, input = hours 0-35, targets = the next 3 hours with an
  observation, times scaled by 48 hours. The benchmark code's tsdm pipeline differs in small details: its time
  scaling (hour / 47 vs. the 35.5 / 48 cutoff) puts the input cutoff at about hour 34.8, and a record's last hourly
  bin is dropped by its parser. These change which points are inputs or targets for some records.
- **Evaluation:** the MSE and MAE are pooled over all observed test targets, as `utils/metrics.py` in the authors'
  code computes them.